# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [10]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [11]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [12]:
# Плоский итератор: превращает Iterable[Iterable[x]] -> Iterable[x]
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [13]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [14]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [15]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [16]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [17]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```

```
map(f) — применили f к каждому входному элементу (k1,v1) → получили поток потоков (k2,v2)*
flatten — превратили поток потоков в один поток (k2,v2)*
groupby(k2) — сгруппировали по ключу k2 → (k2, v2*)*
map(g) — применили g к каждой группе (k2, v2*) → снова поток потоков (k3,v3)*
flatten — получили итоговый плоский поток (k3,v3)*
```

# Примеры

## SQL

```
SELECT age, AVG(social_contacts)
FROM users
WHERE gender = 'female'
GROUP BY age;
```

In [19]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [22]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.2223570036740465)),
 (1, np.float64(2.2223570036740465)),
 (2, np.float64(2.2223570036740465)),
 (3, np.float64(2.2223570036740465)),
 (4, np.float64(2.2223570036740465))]

## Inverted index

In [23]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [26]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [27]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [38]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=REDUCE)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

16 key-value pairs were sent over a network.


[(0, [('', 6)]),
 (1, [('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [39]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.007040519957468017)),
   (None, np.float64(0.05365907195772812)),
   (None, np.float64(0.12244709314882174)),
   (None, np.float64(0.1271320432030989)),
   (None, np.float64(0.14752932137660413)),
   (None, np.float64(0.22747242554766756)),
   (None, np.float64(0.23224423316077447)),
   (None, np.float64(0.25488635027299733)),
   (None, np.float64(0.2560748500377956)),
   (None, np.float64(0.28231681002466824)),
   (None, np.float64(0.2989321112872739)),
   (None, np.float64(0.3649447664786941)),
   (None, np.float64(0.38188058302708494)),
   (None, np.float64(0.4140430820842339)),
   (None, np.float64(0.4345972657147207))]),
 (1,
  [(None, np.float64(0.5718839250092064)),
   (None, np.float64(0.5905852704675472)),
   (None, np.float64(0.6178690653319471)),
   (None, np.float64(0.6222417774072289)),
   (None, np.float64(0.6412345655852962)),
   (None, np.float64(0.6462799584313594)),
   (None, np.float64(0.7274417757201725)),
   (None, np.float64(0.783884580

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [42]:
numbers = [5, 1, 9, 2, 9, -3, 100]

def RECORDREADER():
  for i, val in enumerate(numbers):
      yield (i, val)

def MAP(_, value):
  yield ("", value)

def REDUCE(key, values: Iterator[int]):
  max_val = None
  for val in values:
      if max_val is None or val > max_val:
          max_val = val
  yield (key, max_val)

max_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
max_result

[('', 100)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [44]:
numbers = [5, 1, 9, 2, 9, -3, 10.5]

def RECORDREADER():
  for i, val in enumerate(numbers):
      yield (i, val)

def MAP(_, value):
  yield ("", (value, 1))

def REDUCE(key, values: Iterator[NamedTuple]):
  sum_val = 0.0
  sum_cnt = 0
  for (val, cnt) in values:
      sum_val += val
      sum_cnt += cnt
  if sum_cnt > 0:
      yield (key, sum_val / sum_cnt)
  else:
      yield (key, 0)

avg_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
avg_result

[('', 4.785714285714286)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [48]:
data = [('b', 2), ('a', 1), ('b', 3), ('a', 4), ('c', 5)]

def groupbykey_sort(iterable):
  sorted_list = sorted(list(iterable), key=lambda x: x[0])
  current_key = sorted_list[0][0]
  current_values = []
  for (k, val) in sorted_list:
    if k != current_key: # ключ сменился
      yield (current_key, current_values)
      current_key = k
      current_values = [val]
    else:
      current_values.append(val)
  yield (current_key, current_values)

def MapReduce_sort(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey_sort(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

def RECORDREADER():
  for i, pair in enumerate(data):
    yield (i, pair)

def MAP(i, pair):
  k, val = pair
  yield (k, val)

def REDUCE(k, values):
  # суммируем все значения для ключа k
  sum_key = 0
  for val in values:
    sum_key += val
  yield (k, sum_key)

gb = list(groupbykey_sort(data))
print(gb)
mr_out = list(MapReduce_sort(RECORDREADER, MAP, REDUCE))
print(mr_out)

[('a', [1, 4]), ('b', [2, 3]), ('c', [5])]
[('a', 5), ('b', 5), ('c', 5)]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [63]:
import numpy as np

dups = [1, 2, 2, 3, 3, 3, 10, 10, 5, 5, 5, 5, 7]
maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  def RECORDREADER(split):
      for x in split:
          yield (x, None)
  split_size = int(np.ceil(len(dups) / maps))
  for i in range(0, len(dups), split_size):
      yield RECORDREADER(dups[i:i+split_size])

def MAP(val:int, _):
    yield (val, None)

def REDUCE(val:int, _):
    yield (val, None)

out = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
out = [(pid, list(part)) for (pid, part) in out]
unique_values = [item[0] for _, part in out for item in part]
unique_values

13 key-value pairs were sent over a network.


[2, 10, 1, 3, 5, 7]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [108]:
# Пример данных (id, sex, age)
relation_R = [
    (1, 'female', 25),
    (2, 'male', 30),
    (3, 'female', 25)
]
relation_S = [
    (2, 'male', 30),
    (4, 'male', 40)
]

def RECORDREADER():
  for i, val in enumerate(relation_R):
      yield (i, val)

def MAP_SELECTION(_, value):
  # предикат: возраст > 25
  if value[2] > 25:
        yield (value, value)

def REDUCE_SELECTION(t, values):
    yield (t, values)

select_result = list(MapReduce(RECORDREADER, MAP_SELECTION, REDUCE_SELECTION))
print("Selection (age > 25):", [k for k, v in select_result])

Selection (age > 25): [(2, 'male', 30)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [109]:
def MAP_PROJECTION(_, value):
    projected = (value[1], value[2]) # проекция на атрибуты sex, age
    yield (projected, projected)

def REDUCE_PROJECTION(t, values):
    yield (t, t)

proj_result = list(MapReduce(RECORDREADER, MAP_PROJECTION, REDUCE_PROJECTION))
print("Projection (sex, age):", [k for k, v in proj_result])

Projection (sex, age): [('female', 25), ('male', 30)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [110]:
def RECORDREADER_UNION():
    # читать оба отношения
    for i, val in enumerate(relation_R):
        yield (i, val)
    for i, val in enumerate(relation_S):
        yield (i, val)

def MAP_UNION(_, value):
    yield (value, value)

def REDUCE_UNION(t, values):
    yield (t, t)

union_result = list(MapReduce(RECORDREADER_UNION, MAP_UNION, REDUCE_UNION))
print("Union:", [k for k, v in union_result])

Union: [(1, 'female', 25), (2, 'male', 30), (3, 'female', 25), (4, 'male', 40)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [116]:
def MAP_INTER(_, value):
    yield (value, value)

def REDUCE_INTER(t, values):
  if len(values) == 2:
      yield (t, t)

intersection_result = list(MapReduce(RECORDREADER_UNION, MAP_INTER, REDUCE_INTER))
print("Intersection:", [k for k, v in intersection_result])

Intersection: [(2, 'male', 30)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [120]:
def RECORDREADER_DIFF():
    # читать оба отношения
    for i, val in enumerate(relation_R):
        yield (i, ("R", val))
    for i, val in enumerate(relation_S):
        yield (i, ("S", val))

def MAP_DIFF(_, val_tuple):
    source, data = val_tuple
    yield (data, source)

def REDUCE_DIFF(t, sources):
    if 'R' in sources and 'S' not in sources:
        yield (t, t)

res_diff = list(MapReduce(RECORDREADER_DIFF, MAP_DIFF, REDUCE_DIFF))
print("Difference (R - S):", [k for k, v in res_diff])

Difference (R - S): [(1, 'female', 25), (3, 'female', 25)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [121]:
relation_R_join = [(1, "x"), (2, "y"), (3, "y")] # (a, b)
relation_S_join = [("x", 100), ("y", 200), ("z", 300)] # (b, c)

def RECORDREADER_JOIN():
    for t in relation_R_join:
        yield ("R", t)
    for t in relation_S_join:
        yield ("S", t)

def MAP_JOIN(source, val_tuple):
    if source == "R":
        yield (val_tuple[1], (source, val_tuple[0]))
    elif source == "S":
        yield (val_tuple[0], (source, val_tuple[1]))

def REDUCE_JOIN(b, values):
    list_R = [val[1] for val in values if val[0] == "R"] # a
    list_S = [val[1] for val in values if val[0] == "S"] # c
    for a in list_R:
        for c in list_S:
            yield (None, (a, b, c))

res_join = list(MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN))
print("Natural Join:", [v for k, v in res_join])

Natural Join: [(1, 'x', 100), (2, 'y', 200), (3, 'y', 200)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [125]:
relation_T = [(1, 10, "a"), (1, 5, "b"), (2, 7, "c"), (2, 1, "d"), (3, 100, "f")]

def RECORDREADER_GROUP():
    for i, t in enumerate(relation_T):
        yield (i, t)

def MAP_GROUP(_, t):
    a, b, c = t
    yield (a, b)

def REDUCE_AGGR(a, bs: Iterator[int]):
    x = sum(bs)
    yield (a, x)

group_sum = list(MapReduce(RECORDREADER_GROUP, MAP_GROUP, REDUCE_AGGR))
print("Group summ:", group_sum)

Group summ: [(1, 15), (2, 8), (3, 100)]


### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [135]:
I, J = 4, 5
mat = np.random.rand(I, J)
vec = np.random.rand(J)
# делаем JOIN по j, получаем частичные произведения (i, mij*vj), и вторым MapReduce суммируем по i
def RECORDREADER1():
  for i in range(I):
    for j in range(J):
      yield (("M", i, j), mat[i,j])

  for j in range(J):
    yield (("V", j), vec[j])

def MAP1(key, value:int):
  tag = key[0]
  if tag == "M":
      _, i, j = key
      yield (j, (tag, i, value))
  elif tag == "V":
      _, j = key
      yield (j, (tag, value))

def REDUCE1(j:int, values:Iterator[NamedTuple]):
    m_list = []
    vj = None
    for item in values:
        if item[0] == "M":
            _, i, val = item
            m_list.append((i, val))
        elif item[0] == "V":
            _, vj = item
    for i, val in m_list:
        yield (i, val * vj)

partials = list(MapReduce(RECORDREADER1, MAP1, REDUCE1))

def RECORDREADER2():
    for idx, (i, val) in enumerate(partials):
        yield (idx, (i, val))

def MAP2(_, values):
    i, val = values
    yield (i, val)

def REDUCE2(i, values):
    sum_total = 0.0
    for val in values:
        sum_total += val
    yield (i, sum_total)

In [138]:
reference_solution = np.matmul(mat, vec)
solution = MapReduce(RECORDREADER2, MAP2, REDUCE2)

def asvector(reduce_output):
    reduce_output = list(reduce_output)
    I_ = max(i for (i, _) in reduce_output) + 1
    y = np.empty(shape=(I_,), dtype=float)
    for (i, yi) in reduce_output:
        y[i] = yi
    return y
solution = asvector(solution)
print(reference_solution, solution)
print(np.allclose(reference_solution, solution))

[0.77504484 1.02808655 1.66569982 1.12627276] [0.77504484 1.02808655 1.66569982 1.12627276]
True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [129]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [139]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # проходим по всем строкам i маленькой матрицы
  for i in range(small_mat.shape[0]):
    m_ij = small_mat[i, j]
    # частичное произведение для ячейки (i, k)
    yield ((i, k), m_ij * w)

def REDUCE(key, values):
  (i, k) = key
  summ = 0.0
  for val in values:
      summ += val
  yield ((i, k), summ)

Проверьте своё решение

In [145]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

final_solution = asmatrix(solution)
print(final_solution)
np.allclose(reference_solution, final_solution) # should return true

[[1.76682427 1.21465236 0.78696887 1.58705094 1.85364409 1.11026077
  1.12359357 1.12117101 1.09013836 1.2177129  0.78173918 1.47658604
  1.12318045 1.44519094 0.63892297 1.17811712 1.15300167 0.86081484
  0.34794901 0.54960057 0.83988865 0.812692   1.66269745 0.26875553
  1.13735206 1.63785837 0.81535302 1.4257199  0.86883636 1.15861566
  0.90913394 0.46437604 1.51966879 1.1750434  0.59871333 1.43189598
  0.55353384 1.41623225 1.46250135 1.01059238]
 [0.87878029 0.50421906 0.40519855 0.70891304 0.96212312 0.57568769
  0.62430304 0.5893507  0.40419475 0.50535006 0.31858551 0.82204447
  0.3979335  0.60149216 0.58203111 0.64077596 0.68315355 0.53545165
  0.27841279 0.41507929 0.51792965 0.31963903 0.81444065 0.12641312
  0.5169332  0.75942276 0.40823222 0.87095726 0.26972507 0.79076023
  0.35280182 0.22014311 0.82895383 0.69110437 0.20646363 0.49235078
  0.30367101 0.69909633 0.81465645 0.55825438]]


True

In [141]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [171]:
I, J, K = 3, 4, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def RECORDREADER1():
  for i in range(I):
    for j in range(J):
      yield (("M", i, j), mat_M[i,j])

  for j in range(J):
    for k in range(K):
      yield (("N", j, k), mat_N[j,k])

def MAP1(key, value):
  tag = key[0]
  if tag == "M":
      _, i, j = key
      yield (j, (tag, i, value))
  elif tag == "N":
      _, j, k = key
      yield (j, (tag, k, value))

def REDUCE1(j:int, values:Iterator[NamedTuple]):
  values = list(values)
  list_M = [(i, val) for (src, i, val) in values if src == "M"]
  list_N = [(k, val) for (src, k, val) in values if src == "N"]
  # для фиксированного j перемножаем все пары i из M на k из N
  for i, v in list_M:
      for k, w in list_N:
          yield ((i, k), v * w)

partials = list(MapReduce(RECORDREADER1, MAP1, REDUCE1))

def RECORDREADER2():
    for idx, (i, val) in enumerate(partials):
        yield (idx, (i, val))

def MAP2(_, values):
    i, val = values
    yield (i, val)

def REDUCE2(i, values):
    sum_total = 0.0
    for val in values:
        sum_total += val
    yield (i, sum_total)

In [172]:
# CHECK THE SOLUTION
reference_solution = np.matmul(mat_M, mat_N)
solution = MapReduce(RECORDREADER2, MAP2, REDUCE2)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

final_solution = asmatrix(solution)
print(final_solution)
np.allclose(reference_solution, final_solution)

[[1.19703629 0.92356498 1.32219261 0.98898676 1.1624247 ]
 [0.97200035 0.597911   1.1187893  0.933379   0.80242991]
 [1.08718222 0.75825532 0.82760051 0.7651789  0.98498904]]


True

In [173]:
reduce_output = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))
max(i for ((i,k), vw) in reduce_output)

2

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [178]:
maps = 4
reducers = 2

I, J, K = 4, 3, 6
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def INPUTFORMAT1():
  # split M по строкам
  def RECORDREADER_M(i_from, i_to):
      for i in range(i_from, i_to):
          for j in range(J):
              yield (("M", i, j), mat_M[i, j])
  # split N по столбцам (по k)
  def RECORDREADER_N(k_from, k_to):
      for j in range(J):
          for k in range(k_from, k_to):
              yield (("N", j, k), mat_N[j, k])

  # 2 map-задачи будут читать куски M
  mid_i = I // 2
  yield RECORDREADER_M(0, mid_i)
  yield RECORDREADER_M(mid_i, I)

  # 2 map-задачи будут читать куски N
  mid_k = K // 2
  yield RECORDREADER_N(0, mid_k)
  yield RECORDREADER_N(mid_k, K)

def MAP1(key, value):
  tag = key[0]
  if tag == "M":
      _, i, j = key
      yield (j, (tag, i, value))
  elif tag == "N":
      _, j, k = key
      yield (j, (tag, k, value))

def REDUCE1(j:int, values:Iterator[NamedTuple]):
  values = list(values)
  list_M = [(i, val) for (src, i, val) in values if src == "M"]
  list_N = [(k, val) for (src, k, val) in values if src == "N"]
  # для фиксированного j перемножаем все пары i из M на k из N
  for i, v in list_M:
      for k, w in list_N:
          yield ((i, k), v * w)

partials_dist = [kv for (_pid, it) in MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1, COMBINER=None) for kv in it]

def INPUTFORMAT2():
  global maps
  def RECORDREADER2(split):
    for idx, kv in enumerate(split):
        yield (idx, kv)
  split_size = int(np.ceil(len(partials_dist) / maps)) if partials_dist else 1
  for i in range(0, len(partials_dist), split_size):
      split = partials_dist[i:i+split_size]
      yield RECORDREADER2(split)

def MAP2(_, values):
    ik, val = values
    yield (ik, val)

def REDUCE2(ik, values):
    sum_total = 0.0
    for val in values:
        sum_total += val
    yield (ik, sum_total)

30 key-value pairs were sent over a network.


In [179]:
# CHECK THE SOLUTION
reference_solution = np.matmul(mat_M, mat_N)
solution = [kv for (_pid, it) in MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=None) for kv in it]

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

final_solution = asmatrix(solution)
print(final_solution)
np.allclose(reference_solution, final_solution)

72 key-value pairs were sent over a network.
[[0.37929104 0.27198291 0.28601104 0.50938578 0.58998648 0.63178199]
 [0.45482872 0.85837944 0.71488389 0.29062598 0.24694617 0.668027  ]
 [0.75036835 1.04939713 1.01535588 0.72337459 0.80468163 0.77339695]
 [0.83007513 0.941049   0.91644685 0.91905637 1.03429486 1.11758691]]


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [182]:
m_parts = 3 # сколько recordreaderов для M
n_parts = 5 # сколько recordreaderов для N
maps = m_parts + n_parts
reducers = 2

I, J, K = 4, 3, 6
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

def INPUTFORMAT1():
  def chunk_ranges(n: int, parts: int):
    step = int(np.ceil(n / parts))
    for start in range(0, n, step):
      yield start, min(n, start + step)

  def RECORDREADER_M(i_from, i_to):
      for i in range(i_from, i_to):
        for j in range(J):
          yield (("M", i, j), mat_M[i, j])

  def RECORDREADER_N(k_from, k_to):
      for j in range(J):
        for k in range(k_from, k_to):
          yield (("N", j, k), mat_N[j, k])

  # M режем по строкам на m_parts сплитов
  for (i_from, i_to) in chunk_ranges(I, m_parts):
    yield RECORDREADER_M(i_from, i_to)
  # N режем по столбцам (по k) на n_parts сплитов
  for (k_from, k_to) in chunk_ranges(K, n_parts):
    yield RECORDREADER_N(k_from, k_to)

def MAP1(key, value):
  tag = key[0]
  if tag == "M":
      _, i, j = key
      yield (j, (tag, i, value))
  elif tag == "N":
      _, j, k = key
      yield (j, (tag, k, value))

def REDUCE1(j:int, values:Iterator[NamedTuple]):
  values = list(values)
  list_M = [(i, val) for (src, i, val) in values if src == "M"]
  list_N = [(k, val) for (src, k, val) in values if src == "N"]
  # для фиксированного j перемножаем все пары i из M на k из N
  for i, v in list_M:
      for k, w in list_N:
          yield ((i, k), v * w)

partials_dist = [kv for (_pid, it) in MapReduceDistributed(INPUTFORMAT1, MAP1, REDUCE1, COMBINER=None) for kv in it]

def INPUTFORMAT2():
  global maps
  def RECORDREADER2(split):
    for idx, kv in enumerate(split):
        yield (idx, kv)
  split_size = int(np.ceil(len(partials_dist) / maps)) if partials_dist else 1
  for i in range(0, len(partials_dist), split_size):
      split = partials_dist[i:i+split_size]
      yield RECORDREADER2(split)

def MAP2(_, values):
    ik, val = values
    yield (ik, val)

def REDUCE2(ik, values):
    sum_total = 0.0
    for val in values:
        sum_total += val
    yield (ik, sum_total)

30 key-value pairs were sent over a network.


In [183]:
# CHECK THE SOLUTION
reference_solution = np.matmul(mat_M, mat_N)
solution = [kv for (_pid, it) in MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2, COMBINER=None) for kv in it]

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

final_solution = asmatrix(solution)
print(final_solution)
np.allclose(reference_solution, final_solution)

72 key-value pairs were sent over a network.
[[0.58270334 0.54229487 1.13340505 1.00617725 0.33591633 0.19020071]
 [0.96653161 1.14708304 1.68388554 1.23602946 0.83959324 0.65117655]
 [1.09408151 1.36531735 1.58094386 0.96936957 0.9676534  0.84433325]
 [0.62346952 0.51684163 0.86558685 0.73074351 0.20098402 0.14174094]]


True